# One training step

The dual solves the whole quadratic program at once. A practical alternative walks through the points one at a time. Pegasos does that for the regularized hinge

$$
\frac{\lambda}{2}\|\mathbf{w}\|^2 + \text{average hinge}
$$

This lesson keeps $b = 0$ and updates only $\mathbf{w}$, which is the original simple form. Start at $\mathbf{w} = \mathbf{0}$. At step $t = 1, 2, 3, \ldots$, look at one point $(\mathbf{x}, y)$ and replace

$$
\mathbf{w} \leftarrow \left(1 - \frac{1}{t}\right)\mathbf{w}
$$

If $y\,\mathbf{w}\cdot\mathbf{x} < 1$, add the hinge piece as well:

$$
\mathbf{w} \leftarrow \mathbf{w} + \frac{1}{\lambda t}\, y\mathbf{x}
$$

The first part shrinks the old weight. The second part pushes it toward the side that would have given this point a larger margin. If the point is already outside the street, only the shrinkage happens.


## Three exact steps, $\lambda = 1$

Step $t = 1$, point $(2, 0)$, label $+1$. The zero vector scores $0 < 1$, so the hinge is active. The shrinkage factor is $1 - 1/1 = 0$, and

$$
\mathbf{w}_1 = \frac{1}{1\cdot 1}(+1)(2, 0) = (2, 0)
$$

Step $t = 2$, point $(0, 1)$, label $-1$. The current score is $(2, 0)\cdot(0, 1) = 0$, so $y$ times the score is $0 < 1$. The hinge is active:

$$
\begin{aligned}
\mathbf{w}_2
&= \left(1 - \frac{1}{2}\right)(2, 0) + \frac{1}{2}(-1)(0, 1) \\
&= (1, 0) + (0, -\tfrac{1}{2}) \\
&= \left(1, -\frac{1}{2}\right)
\end{aligned}
$$

Step $t = 3$, point $(2, 0)$ again, label $+1$. Now

$$
y\,\mathbf{w}_2\cdot(2, 0) = 2 > 1
$$

The point is outside the street. Do not add $y\mathbf{x}$. Only shrink:

$$
\mathbf{w}_3 = \left(1 - \frac{1}{3}\right)\left(1, -\frac{1}{2}\right) = \left(\frac{2}{3}, -\frac{1}{3}\right)
$$

These three steps are not the batch solution of the earlier lessons. Two or three updates do not finish the job. They show the rule.


In [1]:
# Three Pegasos steps with fractions. No update when the margin is already > 1.
from fractions import Fraction

lam = Fraction(1)
w = (Fraction(0), Fraction(0))
steps = [
    ((Fraction(2), Fraction(0)), Fraction(1)),
    ((Fraction(0), Fraction(1)), Fraction(-1)),
    ((Fraction(2), Fraction(0)), Fraction(1)),
]
history = []
for t, (x, y) in enumerate(steps, start=1):
    shrunk = ((1 - Fraction(1, t)) * w[0], (1 - Fraction(1, t)) * w[1])
    margin = y * (w[0] * x[0] + w[1] * x[1])
    if margin < 1:
        w = (shrunk[0] + (y * x[0]) / (lam * t), shrunk[1] + (y * x[1]) / (lam * t))
    else:
        w = shrunk
    history.append(w)
assert history[0] == (2, 0)
assert history[1] == (1, Fraction(-1, 2))
assert history[2] == (Fraction(2, 3), Fraction(-1, 3))
for step, weights in enumerate(history, start=1):
    print(step, weights)


1 (Fraction(2, 1), Fraction(0, 1))
2 (Fraction(1, 1), Fraction(-1, 2))
3 (Fraction(2, 3), Fraction(-1, 3))


## The batch line is a different object

The hard-margin line of the three-point example was $\mathbf{w} = (1, 0)$ and $b = -1$. Pegasos as written here has no bias term, and it minimizes a different objective, with $\lambda$ in front of the squared length. Matching $\mathbf{w}_3$ against $(1, 0)$ is not a test that the step was coded correctly. The test is the arithmetic above.


In [2]:
# A fourth step on (0, -1), label -1, still has an active hinge.
from fractions import Fraction

w = (Fraction(2, 3), Fraction(-1, 3))
x = (Fraction(0), Fraction(-1))
y = Fraction(-1)
margin = y * (w[0] * x[0] + w[1] * x[1])
# y * (0 + 1/3) = -1/3, which is below 1, so the hinge fires.
assert margin == Fraction(-1, 3)
t = 4
w = (
    (1 - Fraction(1, t)) * w[0] + (y * x[0]) / t,
    (1 - Fraction(1, t)) * w[1] + (y * x[1]) / t,
)
assert w == (Fraction(1, 2), Fraction(0))
print("after the fourth step", w)


after the fourth step (Fraction(1, 2), Fraction(0, 1))


## A pitfall

The test $y\,\mathbf{w}\cdot\mathbf{x} < 1$ uses the weight from **before** the shrinkage of this step. If you shrink first and then test, you are running a different algorithm. Also, $\lambda$ is not the soft-margin $C$. They both trade fit against a short weight, but they sit in different formulas.

## What to carry forward

Each Pegasos step shrinks $\mathbf{w}$. It adds $y\mathbf{x}/(\lambda t)$ only when the current margin is below $1$. The first three steps with $\lambda = 1$ end at $(2/3,\ -1/3)$.
